In [0]:
# Databricks notebook source
# Czech Fintech — Silver Layer
# Join + typing + SCD Type 2 preparation
# Ultima modifica: 2026-04-03

from pyspark.sql.functions import (
    col, to_date, concat, lit, when, floor, substring,
    year, month, md5, concat_ws, coalesce, row_number
)
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DecimalType, DateType, BooleanType
)
from datetime import datetime

# Config
CATALOG = "czech_fintech"
BRONZE = "bronze"
SILVER = "silver"
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def parse_czech_date(date_str_col):
    """
    Converte YYMMDD (stringa 6 digit) → DateType
    Logica: YY < 50 → 20YY, YY >= 50 → 19YY
    """
    return to_date(
        when(substring(date_str_col, 1, 2).cast(IntegerType()) < 50,
             concat(lit("20"), date_str_col)
        ).otherwise(
            concat(lit("19"), date_str_col)
        ),
        "yyyyMMdd"
    )


In [0]:
# ============================================================================
# STEP 1: LOAD BRONZE TABLES
# ============================================================================

print("=" * 80)
print("STEP 1: LOADING BRONZE TABLES")
print("=" * 80)

trans_bronze = spark.table(f"{CATALOG}.{BRONZE}.transactions")
account_bronze = spark.table(f"{CATALOG}.{BRONZE}.account")
disp_bronze = spark.table(f"{CATALOG}.{BRONZE}.disp")
client_bronze = spark.table(f"{CATALOG}.{BRONZE}.client")
district_bronze = spark.table(f"{CATALOG}.{BRONZE}.district")
card_bronze = spark.table(f"{CATALOG}.{BRONZE}.card")
loan_bronze = spark.table(f"{CATALOG}.{BRONZE}.loan")
order_bronze = spark.table(f"{CATALOG}.{BRONZE}.order")

print(f"✅ transactions: {trans_bronze.count()} rows")
print(f"✅ account: {account_bronze.count()} rows")
print(f"✅ disp: {disp_bronze.count()} rows")
print(f"✅ client: {client_bronze.count()} rows")
print(f"✅ district: {district_bronze.count()} rows")
print(f"✅ card: {card_bronze.count()} rows")
print(f"✅ loan: {loan_bronze.count()} rows")
print(f"✅ order: {order_bronze.count()} rows")


In [0]:
CATALOG = "czech_fintech"
BRONZE = "bronze"
SILVER = "silver"

tables_passthrough = ["account", "client", "card", "disp", "district", "loan", "order"]

for table in tables_passthrough:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SILVER}.{table}")
    
    df = spark.table(f"{CATALOG}.{BRONZE}.{table}").drop("_ingestion_ts")
    
    df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER}.{table}")
    
    count = spark.table(f"{CATALOG}.{SILVER}.{table}").count()
    print(f"✅ {table:<15} → Silver: {count} rows")